# DeepSeek OCR V2 — Self-hosted PDF Text Extraction on Kaggle

This notebook downloads and self-hosts the **DeepSeek-OCR-2** model on Kaggle's free GPU, then uses it to extract text from PDF files and save results as Markdown.

**Workflow:**
1. Clone the project repo from GitHub
2. Install dependencies
3. Download & self-host DeepSeek-OCR-2 model on Kaggle GPU
4. Download PDF data from Google Drive
5. Run OCR pipeline (all inference runs locally on Kaggle GPU)
6. **Save Version** on Kaggle to preserve all outputs

> **Note:** The model (~3 GB) is downloaded from HuggingFace and runs entirely on Kaggle's GPU.
> No data is sent to any external API — all OCR inference is local.

## 1. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/hoangtung386/DeepSeek-OCR-V2-for-PDFs-on-Kaggle.git"
PROJECT_DIR = "/kaggle/working/DeepSeek-OCR-V2-for-PDFs-on-Kaggle"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
OUTPUT_DIR = "/kaggle/working/output"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq poppler-utils > /dev/null

# Install project deps WITHOUT replacing Kaggle's pre-installed PyTorch.
# Kaggle's PyTorch is built for the exact GPU (T4) — overwriting it with a
# PyPI wheel causes "no kernel image" CUDA errors.
!pip install --no-deps -e "."
!pip install "transformers==4.45.2" "Pillow==11.3.0" "pdf2image==1.17.0" \
    "accelerate==1.12.0" "pyyaml==6.0.3" "addict==2.4.0" \
    "huggingface-hub==0.36.2" "tokenizers==0.20.3" "safetensors==0.7.0"

import transformers, torch
print(f"transformers=={transformers.__version__}")
print(f"torch=={torch.__version__} (CUDA {torch.version.cuda})")
print("All dependencies installed.")

## 3. Download & Self-host DeepSeek-OCR-2 Model

Downloads the model weights from HuggingFace Hub and loads them onto Kaggle's GPU.

- **Model:** `deepseek-ai/DeepSeek-OCR-2`
- **Precision:** `float16` (compatible with T4 GPU)
- **Device:** Auto-distributed across available GPUs via `device_map="auto"`
- **All inference runs locally** — no external API calls

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-OCR-2"

print(f"Downloading and loading model: {MODEL_ID}")
print(f"Available GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Force all weights onto GPU 0. DeepSeek-OCR-2's infer() code doesn't handle
# multi-GPU correctly (cross-device tensor errors with device_map="auto").
print("Loading model weights onto GPU 0 (this may take a few minutes)...")
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
).eval()

# Monkey-patch masked_scatter_ to auto-cast source dtype.
# DeepSeek-OCR-2 infer() hardcodes autocast to bfloat16, which T4 doesn't
# support — vision encoder outputs float32 while embeddings stay float16.
_orig_masked_scatter_ = torch.Tensor.masked_scatter_
def _safe_masked_scatter_(self, mask, source):
    if source.dtype != self.dtype:
        source = source.to(self.dtype)
    return _orig_masked_scatter_(self, mask, source)
torch.Tensor.masked_scatter_ = _safe_masked_scatter_

print(f"\nModel self-hosted successfully on Kaggle GPU!")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"GPU memory used: {torch.cuda.memory_allocated(0) / 1024**3:.1f} GB")

## 4. Download PDF Data from Google Drive

Downloads `data.zip` (all PDFs in one file) and extracts into `data/`.

In [ ]:
!pip install -qqq gdown

import gdown
import zipfile
import shutil

GDRIVE_FILE_ID = "1N-R2QV1A-VafbyoGQA_joaIsZNqAXtp8"
ZIP_PATH = "/kaggle/working/data.zip"
EXTRACT_TMP = "/kaggle/working/_extract_tmp"

# Download zip
gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

# Extract to temp directory first
if os.path.exists(EXTRACT_TMP):
    shutil.rmtree(EXTRACT_TMP)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_TMP)
os.remove(ZIP_PATH)

# Find where the PDFs actually are (handles nested folders like data/data/*.pdf)
from pathlib import Path

pdf_files = list(Path(EXTRACT_TMP).rglob("*.[pP][dD][fF]"))
print(f"Found {len(pdf_files)} PDF(s) inside zip")

# Move all PDFs to DATA_DIR (flat)
os.makedirs(DATA_DIR, exist_ok=True)
for pdf in pdf_files:
    dest = os.path.join(DATA_DIR, pdf.name)
    shutil.move(str(pdf), dest)

# Clean up temp
shutil.rmtree(EXTRACT_TMP)

pdf_count = len([f for f in os.listdir(DATA_DIR) if f.lower().endswith(".pdf")])
print(f"Moved {pdf_count} PDF file(s) to {DATA_DIR}")

## 5. Run OCR Pipeline

Passes the pre-loaded model to the pipeline — no re-downloading.
Output writes to `/kaggle/working/output/` for Kaggle's **Output** tab.

In [ ]:
import logging
import warnings
import os

# Suppress noisy warnings from transformers, tensorflow, and absl
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress TF/XLA/cuDNN registration warnings

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

# Build pipeline with pre-loaded model (skips re-download)
from src.config import AppConfig
from src.pipeline import Pipeline

config = AppConfig()
config.paths.data_dir = DATA_DIR
config.paths.output_dir = OUTPUT_DIR

pipeline = Pipeline.__new__(Pipeline)
pipeline.config = config

# Inject the already-loaded model and tokenizer
from src.ocr.engine import DeepSeekOCREngine
pipeline.engine = DeepSeekOCREngine.__new__(DeepSeekOCREngine)
pipeline.engine.config = config.model
pipeline.engine.model = model
pipeline.engine.tokenizer = tokenizer

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Run OCR on all PDFs
pipeline.run()

## 6. Check Results

List all generated Markdown files. Use **Save Version → Save & Run All** to persist outputs.

Files will appear in your notebook's **Output** tab and can be downloaded or used as a Kaggle Dataset.

In [ ]:
md_files = sorted(Path(OUTPUT_DIR).glob("*.md"))
total_size = sum(f.stat().st_size for f in md_files)

print(f"Generated {len(md_files)} Markdown file(s) ({total_size / 1024:.1f} KB total)\n")
for f in md_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:60s} {size_kb:>8.1f} KB")

print(f"\nOutput location: {OUTPUT_DIR}")
print("These files will be saved in Kaggle's Output tab after Save Version.")

## 7. Preview Output

Preview the first 100 lines of the first generated Markdown file.

In [ ]:
if md_files:
    first_file = md_files[0]
    content = first_file.read_text(encoding="utf-8")
    lines = content.splitlines()
    preview = "\n".join(lines[:100])
    print(f"--- Preview: {first_file.name} ({len(lines)} lines total) ---\n")
    print(preview)
    if len(lines) > 100:
        print(f"\n... ({len(lines) - 100} more lines)")
else:
    print("No output files found.")